In [1]:
from pathlib import Path
import os

# Set the project root explicitly so relative config/artifact paths resolve correctly.
PROJECT_ROOT = Path(r"D:\Somesh\NLP_Project").resolve()
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print("Project root:", Path.cwd())

Project root: D:\Somesh\NLP_Project


In [2]:
# Confirm the notebook is running from the project root.
print(Path.cwd())

D:\Somesh\NLP_Project


In [3]:
# Do not use os.chdir("../"): it can move outside the project root.
os.chdir(PROJECT_ROOT)

In [4]:
print("Working directory:", Path.cwd())

Working directory: D:\Somesh\NLP_Project


In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from NLP_Project.constants import *
from NLP_Project.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])

        return ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_path=Path(config.model_path),
            tokenizer_path=Path(config.tokenizer_path),
            metric_file_name=Path(config.metric_file_name),
        )

In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk
import evaluate
import torch
import pandas as pd
from tqdm.auto import tqdm

d:\Somesh\NLP_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    @staticmethod
    def generate_batch_sized_chunks(elements, batch_size):
        """Yield consecutive batches from a sequence."""
        for start in range(0, len(elements), batch_size):
            yield elements[start:start + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=2,
        device=None,
        column_text="dialogue",
        column_summary="summary",
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        if column_text not in dataset.column_names:
            raise ValueError(
                f"Dataset is missing input column {column_text!r}. "
                f"Available columns: {dataset.column_names}"
            )
        if column_summary not in dataset.column_names:
            raise ValueError(
                f"Dataset is missing reference column {column_summary!r}. "
                f"Available columns: {dataset.column_names}"
            )

        inputs_list = dataset[column_text]
        references = dataset[column_summary]
        predictions = []

        model.eval()
        for batch in tqdm(
            self.generate_batch_sized_chunks(inputs_list, batch_size),
            total=(len(inputs_list) + batch_size - 1) // batch_size,
            desc="Generating summaries",
        ):
            encoded = tokenizer(
                batch,
                max_length=1024,
                truncation=True,
                padding=True,
                return_tensors="pt",
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}

            with torch.inference_mode():
                generated = model.generate(
                    input_ids=encoded["input_ids"],
                    attention_mask=encoded["attention_mask"],
                    num_beams=4,
                    max_length=128,
                    length_penalty=0.8,
                )

            predictions.extend(
                tokenizer.batch_decode(
                    generated,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )
            )

        # The evaluate ROUGE metric returns numeric scores (not score.mid.fmeasure).
        return metric.compute(
            predictions=[text.strip() for text in predictions],
            references=[text.strip() for text in references],
            use_stemmer=True,
        )

    def evaluate(self):
        project_root = PROJECT_ROOT
        device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info("Evaluation device: %s", device) if "logger" in globals() else print(
            f"Evaluation device: {device}"
        )

        # Resolve relative paths from the project root.
        def resolve_path(path):
            path = Path(path)
            return path if path.is_absolute() else project_root / path

        model_path = resolve_path(self.config.model_path)
        tokenizer_path = resolve_path(self.config.tokenizer_path)
        configured_data_path = resolve_path(self.config.data_path)

        if not model_path.is_dir():
            raise FileNotFoundError(
                f"Trained model directory not found: {model_path}. "
                "Run model training successfully before evaluation."
            )
        if not tokenizer_path.is_dir():
            raise FileNotFoundError(
                f"Tokenizer directory not found: {tokenizer_path}. "
                "Run model training successfully before evaluation."
            )

        tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))
        model = AutoModelForSeq2SeqLM.from_pretrained(str(model_path)).to(device)

        # Evaluation needs the original dialogue/summary columns. Prefer the
        # configured dataset if it retains them; otherwise use raw ingested SAMSum.
        dataset = None
        if configured_data_path.is_dir():
            candidate = load_from_disk(str(configured_data_path))
            if "test" in candidate and {
                "dialogue", "summary"
            }.issubset(set(candidate["test"].column_names)):
                dataset = candidate

        if dataset is None:
            raw_data_path = project_root / "artifacts" / "data_ingestion" / "samsum_dataset"
            if not raw_data_path.is_dir():
                raise FileNotFoundError(
                    "Could not find a dataset with original dialogue/summary columns. "
                    f"Checked {configured_data_path} and {raw_data_path}."
                )
            dataset = load_from_disk(str(raw_data_path))

        if "test" not in dataset:
            raise ValueError(
                f"Dataset has no 'test' split. Available splits: {list(dataset.keys())}"
            )

        test_dataset = dataset["test"]
        if len(test_dataset) == 0:
            raise ValueError("The test split is empty.")

        # Keep the notebook's small sample for a quick validation run.
        sample_size = min(10, len(test_dataset))
        test_dataset = test_dataset.select(range(sample_size))

        rouge_metric = evaluate.load("rouge")
        scores = self.calculate_metric_on_test_ds(
            dataset=test_dataset,
            metric=rouge_metric,
            model=model,
            tokenizer=tokenizer,
            batch_size=2,
            device=device,
            column_text="dialogue",
            column_summary="summary",
        )

        results = pd.DataFrame([scores], index=["pegasus"])
        metric_path = resolve_path(self.config.metric_file_name)
        metric_path.parent.mkdir(parents=True, exist_ok=True)
        results.to_csv(metric_path, index_label="model")

        print("ROUGE evaluation results:")
        print(results)
        print(f"Saved metrics to: {metric_path}")
        return scores

In [10]:
config = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_config()
evaluator = ModelEvaluation(config=model_evaluation_config)
scores = evaluator.evaluate()

[2026-10-24 11:30:34,290: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-24 11:30:34,292: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-24 11:30:34,293: INFO: common: created directory at: artifacts]
[2026-10-24 11:30:34,294: INFO: common: created directory at: artifacts/model_evaluation]
Evaluation device: cpu


FileNotFoundError: Trained model directory not found: D:\Somesh\NLP_Project\artifacts\model_trainer\pegasus-samsum-model. Run model training successfully before evaluation.